# 0교시 · 선수지식 실습 — 비트, 정밀도, 양자화, NF4, 학습 메모리

> **VLM 경량화 과정 · 0교시(사전 학습) · 실습**
> 실습 환경: **Google Colab (CPU만 사용 — GPU 불필요)**
> 대상: **비전공자 / AI 입문자**

---

## 이 실습의 목표
이 노트북은 GPU 없이, 순수 Python 계산만으로 다음 5가지 개념을 **눈으로 확인**합니다.

1. **비트(bit)** — 숫자를 저장하는 "서랍 크기"
2. **정밀도(precision)** — fp32 / fp16 / int8 / int4가 메모리를 얼마나 차지하는가
3. **양자화(quantization)** — 실수를 정수 칸에 "눌러 담는" 과정과 그 오차
4. **NF4** — 왜 가중치 분포에 맞춰 칸을 비균등하게 배치하면 유리한가
5. **학습 메모리** — 왜 "학습"이 "추론"보다 훨씬 무거운가 (그래디언트·옵티마이저)

> 💡 **이 노트북은 실제 모델을 불러오지 않습니다.** 숫자와 개념만으로 직접 계산해보며 감을 잡는 것이 목적입니다. (실제 모델을 다루는 실습은 다음 노트북 "VLM 구조 실습"에서 진행합니다.)

각 절은 `[코드 실행] → [결과 확인] → [미니 퀴즈]` 순서로 진행됩니다.


## 1. 비트(bit) — 숫자를 담는 "서랍"

컴퓨터의 가장 작은 저장 단위는 **비트(bit)**입니다. 비트 하나는 0 또는 1, 두 가지 상태만 가집니다.

비트가 $n$개 있으면 표현할 수 있는 서로 다른 값의 개수는 $2^n$개입니다.

$$\text{표현 가능한 값의 개수} = 2^{n}$$

예를 들어 4비트는 $2^4 = 16$가지, 8비트는 $2^8 = 256$가지를 표현할 수 있습니다.

아래 코드로 직접 계산해 봅시다.


In [ ]:
# 비트 수에 따라 표현 가능한 값의 개수를 계산
bit_options = [1, 2, 4, 8, 16, 32]

print(f"{'비트 수':<10}{'표현 가능한 값의 개수':>25}")
print("-" * 36)
for bits in bit_options:
    count = 2 ** bits
    # 숫자가 너무 크면 보기 좋게 콤마로 구분해서 출력
    print(f"{bits:<10}{count:>25,}")


비트 수                   표현 가능한 값의 개수
------------------------------------
1                                 2
2                                 4
4                                16
8                               256
16                           65,536
32                    4,294,967,296


### 결과 해석
- 1비트는 단 2가지(0, 1)만 표현합니다 — 동전 던지기 앞/뒤와 같습니다.
- 4비트는 16가지 — 시계 눈금 일부와 비슷한 느�김입니다.
- 8비트는 256가지 — 흑백 사진 한 픽셀의 밝기 단계(0=완전 검정 ~ 255=완전 흰색)와 정확히 같습니다.
- 32비트는 약 43억 가지 — 사람이 직관적으로 느끼기 힘든 수준의 정밀함입니다.

**비유**: 비트는 "서랍의 칸 수"와 같습니다. 칸이 16개뿐인 작은 서랍(4비트)에는 16종류의 물건만 정리할 수 있지만, 칸이 43억 개인 거대한 서랍(32비트)에는 훨씬 세밀하게 정리할 수 있습니다. 다만 서랍이 커질수록 보관 공간(메모리)도 커집니다.

### 미니 퀴즈 1
아래 빈 칸을 채워서, 6비트가 표현할 수 있는 값의 개수를 직접 계산해 보세요.


In [ ]:
# ✏️ 미니 퀴즈: 6비트가 표현 가능한 값의 개수를 계산해 보세요.
# 힌트: 2 ** 6
my_answer = 64  # 여기를 채워보세요 (정답: 64)

if my_answer == 64:
    print("정답입니다! 6비트 = 2^6 = 64가지")
elif my_answer is None:
    print("아직 답을 입력하지 않았습니다. my_answer = 2 ** 6 으로 바꿔보세요.")
else:
    print(f"다시 확인해 보세요. 현재 답: {my_answer}")


정답입니다! 6비트 = 2^6 = 64가지


## 2. 정밀도(precision)와 메모리 — "비트가 늘어나면 메모리도 늘어난다"

AI 모델은 숫자(가중치, weight)들의 거대한 모음입니다. 예를 들어 "4B 모델"이란 모델 안에 약 **44억 개의 숫자**가 들어있다는 뜻입니다 (B = Billion = 10억).

이 숫자 하나를 몇 비트로 저장하느냐에 따라 전체 메모리 사용량이 결정됩니다.

$$\text{전체 메모리(바이트)} = \text{숫자의 개수} \times \frac{\text{비트 수}}{8}$$

(8로 나누는 이유: 1바이트 = 8비트이기 때문입니다. 메모리는 보통 "바이트" 단위로 표시합니다.)

| 정밀도 | 비트 | 1바이트=8비트이므로 바이트/숫자 |
|---|---|---|
| fp32 | 32 | 4바이트 |
| fp16 | 16 | 2바이트 |
| int8 | 8 | 1바이트 |
| int4 | 4 | 0.5바이트 |

아래 코드로 "작은 모델"을 가정하고 직접 계산해 봅시다.


In [ ]:
# ── 정밀도별 메모리 계산 (작은 예시 숫자로 감을 잡기) ──────────
GiB = 1024 ** 3
MiB = 1024 ** 2

def weight_mem(num_params: int, bits: int) -> float:
    '''숫자 개수와 비트 수로 총 메모리(바이트)를 계산'''
    bytes_total = num_params * (bits / 8)
    return bytes_total

# 비유를 위한 작은 모델: 숫자 100만 개(=1M 파라미터)라고 가정
num_params = 1_000_000  # 100만 개

precisions = {"fp32": 32, "fp16": 16, "int8": 8, "int4": 4}

print(f"{'정밀도':<6}{'비트':>6}{'메모리(MiB)':>16}")
print("-" * 36)
for name, bits in precisions.items():
    mem_bytes = weight_mem(num_params, bits)
    mem_mib = mem_bytes / MiB
    print(f"{name:<10}{bits:>6}{mem_mib:>14.3f} MiB")


정밀도       비트        메모리(MiB)
------------------------------------
fp32          32         3.815 MiB
fp16          16         1.907 MiB
int8           8         0.954 MiB
int4           4         0.477 MiB


### 결과 해석
- 똑같은 숫자 100만 개라도, fp32로 저장하면 약 3.8MiB, int4로 저장하면 약 0.48MiB입니다.
- **비트를 절반으로 줄이면 메모리도 정확히 절반**이 됩니다 (fp32 → fp16 → int8 → int4로 갈 때마다 메모리가 반씩 줄어듦).

### 미니 실습
실제 Qwen3-VL-4B 모델(약 44억 개 파라미터)이 fp16과 int4일 때 각각 몇 GiB인지 계산해 보세요.


In [ ]:
# 미니 실습: 실제 모델 크기로 계산해 보기
real_params = 4_437_800_000  # Qwen3-VL-4B의 실제 파라미터 수

fp16_gb = weight_mem(real_params, 16) / GiB
int4_gb = weight_mem(real_params, 4) / GiB

print(f"Qwen3-VL-4B (약 {real_params/1e9:.2f}B 파라미터)")
print(f"  fp16으로 저장하면: {fp16_gb:.2f} GiB")
print(f"  int4로 저장하면:   {int4_gb:.2f} GiB")
print(f"  → int4는 fp16의 약 {fp16_gb/int4_gb:.0f}배 작습니다")


Qwen3-VL-4B (약 4.44B 파라미터)
  fp16으로 저장하면: 8.27 GiB
  int4로 저장하면:   2.07 GiB
  → int4는 fp16의 약 4배 작습니다


## 3. 양자화(Quantization) — "실수를 정수 칸에 눌러 담기"

**양자화**란 연속적인 실수 값을 적은 개수의 정수 칸으로 변환하는 것입니다.

$$\text{양자화(저장)}: \quad q = \text{round}\left(\frac{w}{\text{scale}}\right)$$

$$\text{역양자화(복원)}: \quad \hat{w} \approx q \times \text{scale}$$

- $w$ : 원래의 실수 가중치
- $\text{scale}$ : 정수 한 칸이 대표하는 실수 폭(간격)
- $q$ : 저장되는 정수값
- $\hat{w}$ : 복원된 근사값 (원래 $w$와 정확히 같지 않을 수 있음 — 이 차이가 "양자화 오차")

**비유 — 키를 5cm 단위로만 기록하기**: 학생의 정확한 키(170.3cm)를 "5cm 단위"로만 기록하면 170cm가 됩니다. 종이(메모리)는 아끼지만 0.3cm의 오차가 생깁니다.

아래 코드로 직접 양자화·역양자화를 해보고 오차를 확인합니다.


In [ ]:
import numpy as np

# 가상의 가중치 값 10개 (실제 신경망처럼 0 근처에 몰린 값들)
np.random.seed(42)
weights = np.array([0.13, -0.07, 0.91, -1.20, 0.04, 0.55, -0.33, 1.02, -0.66, 0.18])

def quantize_int4(w, scale):
    '''4bit 정수(-8~7)로 양자화: q = round(w/scale), -8~7 범위로 자르기(clip)'''
    q = np.round(w / scale)
    q = np.clip(q, -8, 7)  # int4 부호있는 정수 범위: -8 ~ 7
    return q.astype(int)

def dequantize(q, scale):
    '''정수를 다시 실수로 복원: w_hat = q * scale'''
    return q * scale

# scale 값 설정: 가중치의 최대 절대값을 7로 나눠서 정함 (간단한 방식)
scale = np.max(np.abs(weights)) / 7
print(f"설정된 scale: {scale:.4f}\n")

q = quantize_int4(weights, scale)
w_hat = dequantize(q, scale)
error = weights - w_hat

print(f"{'원래값(w)':>10}{'정수(q)':>10}{'복원값(w_hat)':>14}{'오차':>10}")
print("-" * 44)
for wi, qi, whi, ei in zip(weights, q, w_hat, error):
    print(f"{wi:>10.3f}{qi:>10d}{whi:>14.3f}{ei:>10.3f}")

print(f"\n평균 절대 오차: {np.mean(np.abs(error)):.4f}")


설정된 scale: 0.1714

    원래값(w)     정수(q)    복원값(w_hat)        오차
--------------------------------------------
     0.130         1         0.171    -0.041
    -0.070         0         0.000    -0.070
     0.910         5         0.857     0.053
    -1.200        -7        -1.200     0.000
     0.040         0         0.000     0.040
     0.550         3         0.514     0.036
    -0.330        -2        -0.343     0.013
     1.020         6         1.029    -0.009
    -0.660        -4        -0.686     0.026
     0.180         1         0.171     0.009

평균 절대 오차: 0.0296


### 결과 해석
- `q` 열은 원래의 실수를 -8~7 사이의 정수로 "눌러 담은" 값입니다.
- `w_hat`은 그 정수를 다시 실수로 복원한 값으로, 원래 값과 똑같지 않습니다.
- `오차` 열을 보면 모든 값에서 약간씩 차이가 생기는 것을 볼 수 있습니다 — 이것이 **양자화로 인한 정밀도 손실**입니다.
- 비트 수를 줄일수록(예: int4 대신 int2) 칸이 더 적어져서 오차가 더 커집니다.

### 미니 실습 — scale을 더 거칠게 바꿔보면?
scale을 2배로 키우면 오차가 어떻게 변하는지 직접 확인해 보세요.


In [ ]:
# 미니 실습: scale을 2배로 키워서 오차 변화 관찰
scale_coarse = scale * 2  # 더 거친(넓은) 칸

q_coarse = quantize_int4(weights, scale_coarse)
w_hat_coarse = dequantize(q_coarse, scale_coarse)
error_coarse = weights - w_hat_coarse

print(f"기존 scale({scale:.4f})의 평균 오차:   {np.mean(np.abs(error)):.4f}")
print(f"2배로 거친 scale({scale_coarse:.4f})의 평균 오차: {np.mean(np.abs(error_coarse)):.4f}")
print()
print("→ scale이 커질수록(칸이 거칠수록) 오차도 커지는 것을 확인할 수 있습니다.")


기존 scale(0.1714)의 평균 오차:   0.0296
2배로 거친 scale(0.3429)의 평균 오차: 0.0876

→ scale이 커질수록(칸이 거칠수록) 오차도 커지는 것을 확인할 수 있습니다.


## 4. 블록 단위(Group-wise) 양자화 — "이상치 한 명이 반 전체를 망치지 않게"

만약 가중치 묶음 안에 유난히 큰 값(이상치, outlier)이 하나 섞여 있으면 어떻게 될까요?
scale은 "최댓값"을 기준으로 정해지기 때문에, 이상치 하나가 전체 칸의 폭을 넓혀버려서 나머지 값들의 정밀도가 다 같이 나빠집니다.

**해결책**: 전체를 한 번에 양자화하지 않고, 작은 묶음(블록)으로 나눠서 **블록마다 scale을 따로** 둡니다.

아래 코드로 "이상치가 있을 때" 전체 양자화와 블록 단위 양자화를 비교합니다.


In [ ]:
# 이상치(outlier)가 섞인 가중치 16개 생성
np.random.seed(0)
normal_part = np.random.normal(0, 0.3, 15)   # 대부분은 0 근처의 작은 값
outlier = np.array([8.0])                     # 딱 하나의 큰 이상치
weights16 = np.concatenate([normal_part, outlier])

print("가중치 16개 (마지막 1개가 이상치):")
print(np.round(weights16, 2))

# (A) 전체를 한 번에 양자화 (이상치 포함 전체 기준 scale)
scale_whole = np.max(np.abs(weights16)) / 7
q_whole = quantize_int4(weights16, scale_whole)
w_hat_whole = dequantize(q_whole, scale_whole)
error_whole = np.mean(np.abs(weights16 - w_hat_whole))

# (B) 블록 단위: 앞 8개와 뒤 8개를 따로 양자화 (이상치는 뒤쪽 블록에만 영향)
block1, block2 = weights16[:8], weights16[8:]
scale1 = np.max(np.abs(block1)) / 7
scale2 = np.max(np.abs(block2)) / 7
q1 = quantize_int4(block1, scale1)
q2 = quantize_int4(block2, scale2)
w_hat_block = np.concatenate([dequantize(q1, scale1), dequantize(q2, scale2)])
error_block = np.mean(np.abs(weights16 - w_hat_block))

print(f"\n[A] 전체 한 번에 양자화 — scale={scale_whole:.3f}, 평균 오차={error_whole:.4f}")
print(f"[B] 블록 단위(8개씩) 양자화 — scale1={scale1:.3f}, scale2={scale2:.3f}, 평균 오차={error_block:.4f}")
print(f"\n→ 블록으로 나누니 평균 오차가 약 {error_whole/error_block:.1f}배 줄었습니다.")
print("   (이상치가 포함된 블록2의 정밀도만 희생되고, 블록1은 영향을 받지 않기 때문)")


가중치 16개 (마지막 1개가 이상치):
[ 0.53  0.12  0.29  0.67  0.56 -0.29  0.29 -0.05 -0.03  0.12  0.04  0.44
  0.23  0.04  0.13  8.  ]

[A] 전체 한 번에 양자화 — scale=1.143, 평균 오차=0.2268
[B] 블록 단위(8개씩) 양자화 — scale1=0.096, scale2=1.143, 평균 오차=0.0736

→ 블록으로 나누니 평균 오차가 약 3.1배 줄었습니다.
   (이상치가 포함된 블록2의 정밀도만 희생되고, 블록1은 영향을 받지 않기 때문)


### 결과 해석
전체를 한 번에 양자화하면 이상치 하나 때문에 scale이 커지고, **이상치와 무관한 나머지 15개 값까지** 정밀도가 나빠집니다.
블록으로 나누면 이상치가 있는 블록만 정밀도가 희생되고, 나머지 블록은 영향을 받지 않습니다.

> 실제 QLoRA/AWQ에서는 보통 64개 또는 128개 단위로 블록을 나눕니다. 여기서는 이해를 돕기 위해 8개로 줄여서 실습했습니다.


## 5. NF4(NormalFloat 4-bit) — "분포에 맞춰 칸을 비균등하게 배치"

신경망의 가중치는 대체로 **0 근처에 많이 몰리고, 0에서 멀어질수록 적어지는 정규분포** 모양을 띕니다.

- 일반 int4: 칸을 **균등한 간격**으로 배치
- **NF4**: 칸을 **0 근처에 촘촘하게, 먼 곳은 듬성듬성** 배치 (정규분포 모양에 맞춤)

같은 16개의 칸(4bit)이라도, "많이 나오는 값"을 더 정확히 표현할 수 있어 평균 오차가 줄어듭니다.

아래 코드로 정규분포를 따르는 가중치에 대해 "균등 격자"와 "비균등(정규분포 맞춤) 격자"를 직접 비교합니다.


In [ ]:
np.linspace(-3, 3, 16)
# array([-3. , -2.6, -2.2, -1.8, -1.4, -1. , -0.6, -0.2,  0.2,  0.6,  1. ,
#         1.4,  1.8,  2.2,  2.6,  3. ])

array([-3. , -2.6, -2.2, -1.8, -1.4, -1. , -0.6, -0.2,  0.2,  0.6,  1. ,
        1.4,  1.8,  2.2,  2.6,  3. ])

In [ ]:
# from scipy.stats import norm
# quantile_points = np.linspace(0.02, 0.98, 16)  # 누적확률 2%~98% 지점들
# nf4_like_levels = norm.ppf(quantile_points)  # 정규분포의 분위수 함수로 칸 위치 결정
# nf4_like_levels = np.clip(nf4_like_levels, -3, 3)
# nf4_like_levels
'''
array([-2.05374891, -1.37865873, -1.0450497 , -0.79950094, -0.59476585,
       -0.41246313, -0.24300697, -0.08029831,  0.08029831,  0.24300697,
        0.41246313,  0.59476585,  0.79950094,  1.0450497 ,  1.37865873,
        2.05374891])
'''

array([-2.05374891, -1.37865873, -1.0450497 , -0.79950094, -0.59476585,
       -0.41246313, -0.24300697, -0.08029831,  0.08029831,  0.24300697,
        0.41246313,  0.59476585,  0.79950094,  1.0450497 ,  1.37865873,
        2.05374891])

In [ ]:
# 하위 2% 점수?
print(norm.ppf(0.02))
# -2.053748910631823

# 중위수(하위 50%) 점수?
print(norm.ppf(0.5))
# 0.0

# 하위 98%(상위 2%) 점수?
print(norm.ppf(0.98))
# 2.0537489106318225

-2.053748910631823
0.0
2.0537489106318225


In [ ]:
# np.linspace(0.01, 0.99, 5)
# # array([0.01 , 0.255, 0.5  , 0.745, 0.99 ])
# q_p = np.linspace(0.01, 0.99, 5)
# levels = norm.ppf(q_p)   # z 값들 >> 기준 격자
# # array([-2.32634787, -0.65883769,  0.        ,  0.65883769,  2.32634787])

# # for i in levels:
# #     print(i)

# levels = np.sort(levels)
# # array([-2.32634787, -0.65883769,  0.        ,  0.65883769,  2.32634787])

# idx = np.searchsorted(levels, 2)
# idx     # np.int64(4)
# # 내 위치가 2이면 인덱스 4 반환
# # >> 내 값(values)이 정렬된 기준선 몇 번째 칸에 끼어 들어가 있는지 인덱스 위치 찾아줌.

# idx = np.clip(idx, 1, len(levels) - 1)
# idx
# # 내 값이 기준선의 맨 왼쪽(최소값) 보다 더 작거나,
# # 맨 오른쪽(최대값)보다 더 크면 안되게 만들어줌.
# # 여기서 1< idx < 최대 마지막 칸

# left = levels[idx - 1]
# print(left)
# right = levels[idx]
# print(right)
# # 양 옆의 기준선 값 알아내기
# # >> 내 값(2)의 바로 왼쪽(작은 쪽) 기준선과 바로 오른쪽(큰 쪽) 기준선이 몇인지 알아내기

# choose_right = (2 - left) > (right - 2)
# np.where(choose_right, right, left)

0.6588376927361878
2.3263478740408408


array(2.32634787)

In [ ]:
# 정규분포를 따르는 가짜 가중치 2000개 생성 (실제 신경망과 비슷한 분포)
np.random.seed(1)
weights_normal = np.random.normal(loc=0.0, scale=1.0, size=2000)
weights_normal = np.clip(weights_normal, -3, 3)  # 너무 극단적인 값은 -3~3으로 제한

# (A) 균등 격자: -3~3 구간을 16개의 균등한 칸으로 나눔 (일반 int4)
uniform_levels = np.linspace(-3, 3, 16)

# (B) 비균등 격자: 정규분포의 분위수(quantile)를 이용해 0 근처에 칸을 더 많이 배치 (NF4와 유사한 원리)
from scipy.stats import norm
quantile_points = np.linspace(0.02, 0.98, 16)  # 누적확률 2%~98% 지점들
nf4_like_levels = norm.ppf(quantile_points)  # 정규분포의 분위수 함수로 칸 위치 결정
nf4_like_levels = np.clip(nf4_like_levels, -3, 3)
# norm.ppf() : percent point function (분위수 함수)
# >> 누적확률(밀도=면적) 알려주면 그 만큼 차지하기 위한 경계선 점수(z값)
# >> 상위/하위 몇 %가 되어야 몇 점인지? 같은 논리

def map_to_nearest_level(values, levels):
    '''각 값을 가장 가까운 격자 레벨에 매핑(반올림과 같은 역할)'''
    levels = np.sort(levels)    # 기준선 정렬
    idx = np.searchsorted(levels, values)
    idx = np.clip(idx, 1, len(levels) - 1)
    left = levels[idx - 1]
    right = levels[idx]
    # 더 가까운 쪽을 선택
    choose_right = (values - left) > (right - values)
    return np.where(choose_right, right, left)
# 연속적인 데이터(values) 정규분포 형태라 어떤 곳(0 근처)은 촘촘하고 어떤 곳은 듬성듬성하기 때문에,
# 해당 데이터를 설계해둔 특정 격자 기준점(levels)를 가지고
# 가장 가까운 곳을 찾아서 거기에 할당하고, 반올림 해주는 필터

recon_uniform = map_to_nearest_level(weights_normal, uniform_levels)
recon_nf4like = map_to_nearest_level(weights_normal, nf4_like_levels)

error_uniform = np.mean(np.abs(weights_normal - recon_uniform))
error_nf4like = np.mean(np.abs(weights_normal - recon_nf4like))

print(f"균등 격자(일반 int4 방식)   평균 오차: {error_uniform:.4f}")
print(f"비균등 격자(NF4 방식)      평균 오차: {error_nf4like:.4f}")
print(f"\n→ NF4 방식이 균등 격자보다 오차가 약 {(1 - error_nf4like/error_uniform)*100:.1f}% 더 적습니다.")
print("   (가중치가 0 근처에 몰려있다는 사실을 활용했기 때문)")


균등 격자(일반 int4 방식)   평균 오차: 0.0986
비균등 격자(NF4 방식)      평균 오차: 0.0782

→ NF4 방식이 균등 격자보다 오차가 약 20.7% 더 적습니다.
   (가중치가 0 근처에 몰려있다는 사실을 활용했기 때문)


### 결과 해석
같은 16개의 칸(4bit)을 쓰더라도, **데이터가 몰려있는 구간에 칸을 더 많이 배치**하면 평균 오차가 줄어듭니다.
이것이 NF4가 일반 int4보다 신경망 가중치 양자화에 더 적합한 이유입니다.

> 실제 NF4는 이론적으로 유도된 정확한 12개의 양자화 레벨(4bit=16개 중 부호 대칭 등을 고려)을 사용하지만, 여기서는 "분포에 맞춰 칸을 배치한다"는 핵심 원리만 체험하기 위해 간소화했습니다.

### 미니 퀴즈 2
다음 문장의 빈칸을 채워보세요 (코드로 확인).

"가중치가 ( )분포를 따르고 ( ) 근처에 많이 몰려 있기 때문에, NF4는 그 구간에 칸을 더 ( )하게 배치한다."


In [1]:
# 미니 퀴즈 2 정답 확인
answer1 = "정규"  # 정규
answer2 = "0"  # 0
answer3 = "촘촘"  # 촘촘

if answer1.strip() == "정규" and answer2.strip() == "0" and answer3.strip() in ["촘촘", "조밀"]:
    print("정답입니다!")
else:
    print("빈칸을 채운 뒤 다시 실행해 보세요. (정답: 정규 / 0 / 촘촘)")


정답입니다!


## 6. 학습(training) vs 추론(inference) — 왜 학습이 더 무거운가

- **추론**: 완성된 가중치로 계산만 함 (가중치 메모리만 필요)
- **학습**: 가중치를 더 좋게 수정함 → 가중치 외에 **그래디언트**와 **옵티마이저 상태**가 추가로 필요

| 구성 | 정밀도 | 바이트/숫자 | 설명 |
|---|---|---|---|
| 가중치(fp16) | fp16 | 2 | 계산의 기준값 |
| 그래디언트(fp16) | fp16 | 2 | "어느 방향으로 바꿀지" |
| 옵티마이저 모멘텀(fp32) | fp32 | 4 | "최근 이동 방향의 평균" |
| 옵티마이저 분산(fp32) | fp32 | 4 | "이동의 흔들림 정도" |
| 마스터 가중치(fp32) | fp32 | 4 | 혼합정밀 학습용 정밀 사본 |

$$\text{전체 파인튜닝 메모리} \approx \text{파라미터 수} \times 16\text{바이트} \quad (2+2+4+4+4=16)$$

아래 코드로 직접 계산해서, 추론 대비 학습이 몇 배 더 무거운지 확인합니다.


In [1]:
# ── 추론 vs 학습 메모리 비교 ──────────────────────────
GiB = 1024 ** 3
P = 4_437_800_000  # Qwen3-VL-4B 파라미터 수

def gib(byte_count):
    return byte_count / GiB

# 추론: 가중치(fp16, 2바이트)만 필요
inference_mem = P * 2

# 전체 파인튜닝: 가중치 + 그래디언트 + 옵티마이저(모멘텀,분산) + 마스터가중치
full_train_mem = (
    P * 2 +  # fp16 가중치
    P * 2 +  # fp16 그래디언트
    P * 4 +  # fp32 모멘텀
    P * 4 +  # fp32 분산
    P * 4    # fp32 마스터 가중치
)

print(f"추론(inference)만 할 때:        {gib(inference_mem):>7.2f} GiB")
print(f"전체 파인튜닝(full fine-tuning): {gib(full_train_mem):>7.2f} GiB")
print(f"\n→ 학습은 추론보다 약 {full_train_mem/inference_mem:.1f}배 더 많은 메모리가 필요합니다.")
print(f"   (활성값(activation)은 아직 포함하지 않은 수치입니다)")


추론(inference)만 할 때:           8.27 GiB
전체 파인튜닝(full fine-tuning):   66.13 GiB

→ 학습은 추론보다 약 8.0배 더 많은 메모리가 필요합니다.
   (활성값(activation)은 아직 포함하지 않은 수치입니다)


### 결과 해석
같은 모델인데도 "그냥 답을 생성하는 것(추론)"과 "더 똑똑하게 만드는 것(학습)"은 메모리 요구량이 8배나 차이 납니다.
이 차이를 메우기 위해 등장한 기법이 바로 다음 노트북에서 다룰 **LoRA**와 **QLoRA**입니다 — "그래디언트·옵티마이저가 필요한 부분을 1%도 안 되는 작은 어댑터로 줄이자"는 아이디어입니다.

### 미니 실습 — LoRA 어댑터를 가정해서 비교해보기
만약 전체 파라미터의 0.5%만 학습한다면(나머지는 그래디언트·옵티마이저 불필요), 메모리가 얼마나 줄어들까요?


In [2]:
# 미니 실습: LoRA처럼 일부만 학습 대상일 때의 메모리
LORA_RATIO = 0.005  # 전체의 0.5%만 학습
P_adapter = int(P * LORA_RATIO)

# 학습 대상이 아닌 나머지(base)는 4bit로 동결(가중치만 필요, 그래디언트/옵티마이저 없음)
base_frozen_mem = P * 0.5  # int4 = 0.5바이트/param

# 학습 대상(어댑터)만 그래디언트 + 옵티마이저 필요 (fp16 기준)
adapter_mem = (
    P_adapter * 2 +  # fp16 가중치
    P_adapter * 2 +  # fp16 그래디언트
    P_adapter * 4 +  # fp32 모멘텀
    P_adapter * 4    # fp32 분산
)

qlora_like_mem = base_frozen_mem + adapter_mem

print(f"학습 대상 파라미터: 전체 {P/1e9:.2f}B 중 {P_adapter/1e6:.1f}M개 (={LORA_RATIO*100:.1f}%)")
print(f"전체 파인튜닝:        {gib(full_train_mem):>7.2f} GiB")
print(f"LoRA 방식(가정):       {gib(qlora_like_mem):>7.2f} GiB")
print(f"\n→ 약 {full_train_mem/qlora_like_mem:.0f}배 메모리를 절감했습니다.")


학습 대상 파라미터: 전체 4.44B 중 22.2M개 (=0.5%)
전체 파인튜닝:          66.13 GiB
LoRA 방식(가정):          2.31 GiB

→ 약 29배 메모리를 절감했습니다.


## 7. 종합 정리 — 오늘 실습에서 확인한 것

| 절 | 확인한 내용 |
|---|---|
| 1. 비트 | 비트 수가 늘어나면 표현 가능한 값이 $2^n$으로 폭증 |
| 2. 정밀도와 메모리 | 메모리 = 파라미터 수 × (비트/8), 비트 절반 → 메모리 절반 |
| 3. 양자화 | `q=round(w/scale)`로 실수를 정수 칸에 매핑, 오차 발생 |
| 4. 블록 단위 양자화 | 이상치의 영향을 블록 안에만 가두어 전체 오차를 줄임 |
| 5. NF4 | 0 근처에 칸을 촘촘히 배치 → 같은 비트로도 오차 감소 |
| 6. 학습 vs 추론 | 학습은 그래디언트·옵티마이저까지 필요해 추론보다 8배 무거움 |

> **체크포인트**: 다음 노트북(VLM 구조 실습)으로 넘어가기 전에, 아래 질문에 스스로 답해볼 수 있다면 준비가 된 것입니다.
> 1. 16비트를 4비트로 줄이면 메모리는 몇 분의 1이 되는가?
> 2. 양자화에서 `scale`은 어떤 역할을 하는가?
> 3. NF4가 일반 int4보다 신경망에 유리한 이유는 무엇인가?
> 4. 학습이 추론보다 메모리를 더 쓰는 이유 2가지는?

다음 노트북: **VLM 구조 실습 — 이미지가 패치를 거쳐 토큰이 되는 과정**
